## K-means and Hierarchical Clustering

We'll explore a few examples of clustering today.  The following sites may prove helpful:
- [https://scikit-learn.org/1.5/modules/generated/sklearn.cluster.KMeans.html](https://scikit-learn.org/1.5/modules/generated/sklearn.cluster.KMeans.html)
- [https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.linkage.html)
- [https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html](https://docs.scipy.org/doc/scipy/reference/generated/scipy.cluster.hierarchy.dendrogram.html)

In [ ]:
# As always, we start with our imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_blobs # used just to make some example data
from sklearn.cluster import KMeans # <-- NEW!!! needed to build our k-means model
from sklearn.metrics import accuracy_score
from scipy.cluster.hierarchy import dendrogram, linkage # <-- NEW!!! needed to build our hierarchical model
from sklearn.datasets import load_iris

# for dbscan example
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import DBSCAN

In [ ]:
# just making some sample data for us to practice with
X, y_true = make_blobs(n_samples=300, centers=4,
                       cluster_std=0.60, random_state=0)
plt.scatter(X[:, 0], X[:, 1], s=50)

We can see several clear clusters via visual inspection. Can k-means identify them?

Note: why am I am not using test and training data here? 

In [ ]:
# Let's create and fit the model



In [ ]:
# to address possible error
import os
os.environ["OMP_NUM_THREADS"] = "2"

In [ ]:
# now let's get the labels for each data point (predict)


In [ ]:
# this provides the labels for each of the 300 samples


Let's view what it predicted via plotting. 

In [ ]:
# we can access each like so


In [ ]:
# we can also access inertia


In [ ]:
# let's compare to another value for K
kmeans = KMeans(n_clusters=13)
kmeans.fit(X)
y = kmeans.predict(X)

Let's plot is again to visualize what we have. 

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y, s=50)

centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c="black", s=200, alpha=0.5)

In [ ]:
# inertia?
kmeans.inertia_

Which is actually better?

## Another Example

We'll load the digits dataset. This has handwritten digits from 0 to 9. It stores pixels values for each observation (image). Each image is actually an 8x8 pixel grayscale image, and represented as a 64 dimensional feature vector. Each number represents the pixel intensity. Our goal is to group similar handwritten digits together. 

See https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits for more about this dataset. And see https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html to see how this is imported.

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits(as_frame=True)
digits.data.shape

Here is a quick breakdown of how the feature values work in this dataset:
* The Scale: Each sample is an 8x8 image of a handwritten digit. The features represent individual pixels, and their values range strictly from 0 to 16.
* 0 (Minimum): Represents the background (completely white/empty space).
* 16 (Maximum): Represents the highest amount of ink or pressure (completely dark/black).
* 1 to 15: Represents intermediate shades of gray (anti-aliasing or lighter strokes).

In [ ]:
digits.data

In [ ]:
# Show plot in grayscale
plt.gray()
# Display a 2D array as a matrix in a new figure window.
plt.matshow(digits.images[79])

In [ ]:
# we know this contains 10 digits... let's see if k-means can find them
kmeans = KMeans(n_clusters=10, random_state=0)
clusters = kmeans.fit_predict(digits.data) # note that this fits and predicts all in one step!!!! It returns array containing cluster values.
kmeans.cluster_centers_.shape

Let's check out the centers visually, but we have to transform them a bit to make this more understandable.  

Note that if we just look at centers (as shown below), this isn't super informative. 

In [ ]:
kmeans.cluster_centers_

In [ ]:
# get ready for plotting
fig, ax = plt.subplots(2, 5, figsize=(8, 3))

# Take the 10 K-Means cluster centers (each of length 64) and reshape them into 10 images of size 8×8 for visualization or analysis
# cluster centers is each center's coordinates. With 64 features, this means a cluster center has 64 values in a coordinate
# use 8 x 8 to represent the original image dimensions 

# For each subplot (axi) and its corresponding cluster center image (center), display that image on that subplot.
centers = kmeans.cluster_centers_.reshape(10, 8, 8)  
for axi, center in zip(ax.flat, centers):
    axi.set(xticks=[], yticks=[]) # removes tick marks for cleaner output
    axi.imshow(center, interpolation='nearest', cmap=plt.cm.binary) # show image; place image in center of subplot & present as black and white colormap 

## Hierarchical Clustering 

Let's use our iris data set to see how it clusters the data without the label (target). Then we'll use the target to see if it is accurate.

In [ ]:
# load iris and get a sample of the data
iris = load_iris(as_frame=True)
df = iris.frame
df = df.sample(frac=0.20, replace=True, random_state=1)

In [ ]:
# let's see what we have


Let's build a pretty simple model using just the sepal length and width. 

In [ ]:
# select just sepal length and width


In [ ]:
# need labels in list to pass to plot below


In [ ]:
# Create the dendrogram


# Plot the dendrogram
plt.ylabel('Width Differences', fontsize=14)
plt.yticks(np.arange(0, .7, step=.05))
plt.xlabel('Species', fontsize=14)
plt.title('Single linkage clustering', fontsize=16)
plt.show()

How did it do?

## DBSCAN Example

Let's begin by loading the data. We don't want to include customer score when we do clustering so, we'll select the features we need.

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
X = df[['Age', 'AnnualIncome', 'SpendingScore']]
X

Distance-based methods should be scaled first. We'll use MinMax scaler this time.

In [ ]:
scaler = MinMaxScaler(feature_range=(0,10))
model = scaler.fit(X)
scaled_X = model.transform(X) # gives a ndarry
# put this back into a dataframe
X = pd.DataFrame(scaled_X, columns=X.columns)

Now we can create our DBSCAN model.
We'll specify max distance between samples for them to be consider in the same neighborhood (esp). This is the most important parameter to consider.
We'll also specify the min number of samples needed for a point to be considered a core point. 


Let's now add cluster labels to each observation in the ORIGINAL dataset.

In [ ]:
df.loc[:,'Cluster'] = DBSCANmodel.labels_ 
df['Cluster'] = df['Cluster'].astype(int)
df

Let's just see how many observations belong to each cluster.

In [ ]:
df.value_counts("Cluster").to_frame()

In [ ]:
# get outliers
outliers = df[df['Cluster']==-1]

# get non-outliers
non_outliers = df[df['Cluster']!=-1]

fig, (axes) = plt.subplots(1,2, figsize=(12,5))

# plot the clusters
sns.scatterplot(x='AnnualIncome', y='SpendingScore', data=non_outliers, hue='Cluster', ax=axes[0], s=50)
sns.scatterplot(x='Age', y='SpendingScore', data=non_outliers, hue='Cluster', ax=axes[1], s=50)

# add outliers to each plot
axes[0].scatter(outliers['AnnualIncome'], outliers['SpendingScore'], s=50, label='Outlier', c="y")
axes[1].scatter(outliers['Age'], outliers['SpendingScore'], s=50, label='Outlier', c="y")

# add legend
axes[0].legend()
axes[1].legend()

# see https://seaborn.pydata.org/generated/seaborn.move_legend.html
sns.move_legend(axes[0], "upper right", title="Cluster")
sns.move_legend(axes[1], "upper right", title="Cluster")

plt.show()

In [ ]:
What does this tell us? 